Import potrzebnych bibliotek

In [140]:
import pandas as pd
import numpy as np

Wczytanie danych

In [141]:
df = pd.read_csv("auction_results_color_svd.csv")

df.head()

,ARTIST,TECHNIQUE,SIGNATURE,CONDITION,TOTAL DIMENSIONS,YEAR,Colorfulness Score,SVD Entropy,PRICE
0,218,-1.295300,1,2,-0.157723,-1.039766,51.632554,5.453204,150
1,101,-0.122087,2,2,-0.442668,-0.580107,161.631656,6.154763,270
2,274,-0.122087,2,2,0.263423,-0.626073,117.464780,6.908661,360
3,354,3.397553,0,2,-0.827075,-0.488176,164.609302,6.986244,343
4,354,-0.122087,2,2,-0.145178,0.431142,91.023011,5.859255,150


In [142]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22110 entries, 0 to 22109
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ARTIST              22110 non-null  int64  
 1   TECHNIQUE           22110 non-null  float64
 2   SIGNATURE           22110 non-null  int64  
 3   CONDITION           22110 non-null  int64  
 4   TOTAL DIMENSIONS    22110 non-null  float64
 5   YEAR                22110 non-null  float64
 6   Colorfulness Score  22110 non-null  float64
 7   SVD Entropy         22110 non-null  float64
 8   PRICE               22110 non-null  int64  
dtypes: float64(5), int64(4)
memory usage: 1.5 MB


Dopasowanie typów zmiennych

In [143]:
zmienne_kategoryczne =  ['ARTIST', 'TECHNIQUE', 'SIGNATURE', 'CONDITION']
bloki_kategoryczne = []
for zmienna in zmienne_kategoryczne:
    kody = df[zmienna].astype('category').cat.codes.to_numpy(dtype=np.int32)
    liczba_klas = np.max(kody) + 1
    one_hot = np.eye(liczba_klas)[kody]
    bloki_kategoryczne.append(one_hot)


X_kat_cale = np.hstack(bloki_kategoryczne)


Dane treningowe i testowe

In [144]:
np.random.seed(42)
df_train = df.sample(frac=0.8, random_state=42)
df_test = df.drop(df_train.index)

indeksy_train = df_train.index
indeksy_test = df_test.index

X_kat_train = X_kat_cale[indeksy_train]
X_kat_test = X_kat_cale[indeksy_test]

Normalizacja zmiennych

In [145]:
zmienne_liczbowe = ["TOTAL DIMENSIONS", "YEAR", "Colorfulness Score", "SVD Entropy"]
srednia = df_train[zmienne_liczbowe].mean()
odchylenie = df_train[zmienne_liczbowe].std()

df_train[zmienne_liczbowe] = (df_train[zmienne_liczbowe] - srednia) / odchylenie
df_test[zmienne_liczbowe] = (df_test[zmienne_liczbowe] - srednia) / odchylenie

Przejście na tablice NumPy

In [146]:
X_num_train = df_train[zmienne_liczbowe].to_numpy(dtype=np.float32)
X_num_test = df_test[zmienne_liczbowe].to_numpy(dtype=np.float32)

X_train = np.hstack([X_kat_train, X_num_train], dtype=np.float32)
X_test = np.hstack([X_kat_test, X_num_test], dtype=np.float32)

srednia_cena = df_train['PRICE'].mean()
odchylenie_cena = df_train['PRICE'].std()

y_train = ((df_train['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)
y_test = ((df_test['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)




Funkcja do obliczania prawdziwych cen

In [147]:
def prawdziwa_cena(znormalizowana_cena):
    return znormalizowana_cena * odchylenie_cena + srednia_cena

Tworzenie Dense Layer

In [148]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases
        
    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        self.dinputs = np.dot(dvalues, self.weights.T)

Funkcje aktywacji

In [149]:
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
        
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

class Activation_Linear:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = inputs
    
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()

Funkcja straty

In [150]:
class Loss:
    def calculate(self, output, y):
        sample_losses = self.forward(output, y)
        data_loss = np.mean(sample_losses)
        return data_loss
    
class Loss_MSE(Loss):
    def forward(self, y_pred, y_true):
        sample_losses = np.mean((y_pred - y_true) ** 2, axis=-1)
        return sample_losses
    
    def backward(self, dvalues, y_true):
        liczba_probek = len(dvalues)
        self.dinputs = -2 * (y_true - dvalues) / liczba_probek



Optymalizator SGD

In [151]:
class Optimizer_SGD:
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
    
    def update_params(self, layer):
        layer.weights -= self.learning_rate * layer.dweights
        layer.biases -= self.learning_rate * layer.dbiases

Inicjalizacja sieci i optymalizatora

In [152]:
liczba_cech = X_train.shape[1]

dense1 = Layer_Dense(liczba_cech, 64)
activation1 = Activation_ReLU()

dense2 = Layer_Dense(64, 32)
activation2 = Activation_ReLU()

dense3 = Layer_Dense(32, 1)
activation3 = Activation_Linear()

loss_function = Loss_MSE()


optimizer = Optimizer_SGD(learning_rate=0.1)

Pętla ucząca

In [153]:
"""
epoki = 100

print("Rozpoczynamy trening sieci!\n")

for epoch in range(epoki + 1):
    
    # 1. Przejście w przód (Forward pass)
    dense1.forward(X_train)
    activation1.forward(dense1.output)
    
    dense2.forward(activation1.output)
    activation2.forward(dense2.output)
    
    dense3.forward(activation2.output)
    activation3.forward(dense3.output)

    # 2. Obliczenie błędu
    loss = loss_function.calculate(activation3.output, y_train)

    # 3. Propagacja wsteczna (Backward pass)
    loss_function.backward(activation3.output, y_train)
    
    activation3.backward(loss_function.dinputs)
    dense3.backward(activation3.dinputs)
    
    activation2.backward(dense3.dinputs)
    dense2.backward(activation2.dinputs)
    
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # 4. Aktualizacja wag
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.update_params(dense3)

    # 5. Wyświetlanie postępów
    if epoch % 10 == 0:
        print(f"Epoka: {epoch:3} | Błąd (MSE): {loss:.5f}")

print("\nTrening zakończony!")
"""


'\nepoki = 100\n\nprint("Rozpoczynamy trening sieci!\n")\n\nfor epoch in range(epoki + 1):\n    \n    # 1. Przejście w przód (Forward pass)\n    dense1.forward(X_train)\n    activation1.forward(dense1.output)\n    \n    dense2.forward(activation1.output)\n    activation2.forward(dense2.output)\n    \n    dense3.forward(activation2.output)\n    activation3.forward(dense3.output)\n\n    # 2. Obliczenie błędu\n    loss = loss_function.calculate(activation3.output, y_train)\n\n    # 3. Propagacja wsteczna (Backward pass)\n    loss_function.backward(activation3.output, y_train)\n    \n    activation3.backward(loss_function.dinputs)\n    dense3.backward(activation3.dinputs)\n    \n    activation2.backward(dense3.dinputs)\n    dense2.backward(activation2.dinputs)\n    \n    activation1.backward(dense2.dinputs)\n    dense1.backward(activation1.dinputs)\n\n    # 4. Aktualizacja wag\n    optimizer.update_params(dense1)\n    optimizer.update_params(dense2)\n    optimizer.update_params(dense3)\n\n

In [ ]:
epoki = 100           
batch_size = 256      

print("Rozpoczynamy szybki trening na paczkach!\n" + "-"*30)

for epoch in range(epoki + 1):
    
    loss_epoki = 0
    ilosc_paczek = 0
    
    
    for start_idx in range(0, len(X_train), batch_size):
        
        
        end_idx = start_idx + batch_size
        X_batch = X_train[start_idx:end_idx]
        y_batch = y_train[start_idx:end_idx]
        
        
        dense1.forward(X_batch)
        activation1.forward(dense1.output)
        
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)

        
        loss = loss_function.calculate(activation3.output, y_batch)
        loss_epoki += loss
        ilosc_paczek += 1

        
        loss_function.backward(activation3.output, y_batch)
        
        activation3.backward(loss_function.dinputs)
        dense3.backward(activation3.dinputs)
        
        activation2.backward(dense3.dinputs)
        dense2.backward(activation2.dinputs)
        
        activation1.backward(dense2.dinputs)
        dense1.backward(activation1.dinputs)

        
        optimizer.update_params(dense1)
        optimizer.update_params(dense2)
        optimizer.update_params(dense3)

    
    sredni_blad_mse = loss_epoki / ilosc_paczek
    
    if epoch % 10 == 0:
        print(f"Epoka: {epoch:3} | Średni błąd (MSE): {sredni_blad_mse:.5f}")

print("-" * 30 + "\nTrening zakończony!")

Rozpoczynamy szybki trening na paczkach!
------------------------------
Epoka:   0 | Średni błąd (MSE): 0.99001
Epoka:   5 | Średni błąd (MSE): 0.98967
Epoka:  10 | Średni błąd (MSE): 0.87414
Epoka:  15 | Średni błąd (MSE): 0.69979
Epoka:  20 | Średni błąd (MSE): 0.60827
Epoka:  25 | Średni błąd (MSE): 0.55275
Epoka:  30 | Średni błąd (MSE): 0.51848
Epoka:  35 | Średni błąd (MSE): 0.48684
Epoka:  40 | Średni błąd (MSE): 0.46142
Epoka:  45 | Średni błąd (MSE): 0.44222
Epoka:  50 | Średni błąd (MSE): 0.42687
Epoka:  55 | Średni błąd (MSE): 0.41422
Epoka:  60 | Średni błąd (MSE): 0.40452
Epoka:  65 | Średni błąd (MSE): 0.39770
Epoka:  70 | Średni błąd (MSE): 0.39079
Epoka:  75 | Średni błąd (MSE): 0.38019
Epoka:  80 | Średni błąd (MSE): 0.37024
Epoka:  85 | Średni błąd (MSE): 0.36312
Epoka:  90 | Średni błąd (MSE): 0.35157
Epoka:  95 | Średni błąd (MSE): 0.34457
Epoka: 100 | Średni błąd (MSE): 0.33567
------------------------------
Trening zakończony!


In [155]:
print("=== TESTOWANIE SIECI NA NIEZNANYCH DANYCH (X_test) ===\n")

dense1.forward(X_test)
activation1.forward(dense1.output)

dense2.forward(activation1.output)
activation2.forward(dense2.output)

dense3.forward(activation2.output)
activation3.forward(dense3.output)

test_loss = loss_function.calculate(activation3.output, y_test)

print(f"Błąd (MSE) na zbiorze TRENINGOWYM wynosił: {sredni_blad_mse:.5f}")
print(f"Błąd (MSE) na zbiorze TESTOWYM wynosi:       {test_loss:.5f}")
print("-" * 60)


print("5 przykładowych wycen dla zupełnie nowych obrazów:\n")

for i in range(5):
    
    wymyslona_znormalizowana = activation3.output[i].item()
    prawdziwa_znormalizowana = y_test[i].item()
    
    wymyslona_prawdziwa = prawdziwa_cena(wymyslona_znormalizowana)
    prawdziwa_w_dolarach = prawdziwa_cena(prawdziwa_znormalizowana)
    
    pomylka = abs(prawdziwa_w_dolarach - wymyslona_prawdziwa)
    
    print(f"Obraz {i+1} | Wyliczona: {wymyslona_prawdziwa:12.2f} | Prawdziwa: {prawdziwa_w_dolarach:12.2f} | Pomyłka: {pomylka:10.2f}")

=== TESTOWANIE SIECI NA NIEZNANYCH DANYCH (X_test) ===

Błąd (MSE) na zbiorze TRENINGOWYM wynosił: 0.33567
Błąd (MSE) na zbiorze TESTOWYM wynosi:       0.36504
------------------------------------------------------------
5 przykładowych wycen dla zupełnie nowych obrazów:

Obraz 1 | Wyliczona:         2.49 | Prawdziwa:        55.00 | Pomyłka:      52.51
Obraz 2 | Wyliczona:        -6.17 | Prawdziwa:        55.00 | Pomyłka:      61.17
Obraz 3 | Wyliczona:        87.88 | Prawdziwa:       124.00 | Pomyłka:      36.12
Obraz 4 | Wyliczona:       101.67 | Prawdziwa:        71.00 | Pomyłka:      30.67
Obraz 5 | Wyliczona:        52.88 | Prawdziwa:        79.00 | Pomyłka:      26.12
